In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

In [ ]:
# Task 2: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:

feature_cols = [c for c in df.columns if c != target_col]
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])


In [ ]:
# Task 5: Write your code here:

print(df[target_col].value_counts(normalize=True))


In [ ]:
# Task 1: Write your code here:
X = df[feature_cols].values
y = df[target_col].values

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
models = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = CatBoostClassifier(
        depth=6,
        learning_rate=0.05,
        n_estimators=500,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=False,
        random_state=42
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    models.append(model)

print("F1 scores:", f1_scores)
print("Average F1:", np.mean(f1_scores))

best_model = models[np.argmax(f1_scores)]


In [ ]:
# Task 1: Write your code here:# Part 4 -------------------------------------------------
importances = best_model.get_feature_importance()
sorted_idx = np.argsort(importances)[::-1]
sorted_features = np.array(feature_cols)[sorted_idx]

plt.figure(figsize=(8,5))
plt.bar(range(20), importances[sorted_idx][:20])
plt.xticks(range(20), sorted_features[:20], rotation=90)
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()




In [ ]:
# Task 2: Write your code here:
golden_feature = sorted_features[0]
print("Golden feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here:
X_golden = df[[golden_feature]].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_golden = []

for train_idx, val_idx in skf.split(X_golden, y):
    X_train, X_val = X_golden[train_idx], X_golden[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]


print("Golden-only F1 scores:", f1_golden)
print("Average F1 (golden only):", np.mean(f1_golden))
